# MRIQC: MRI Quality Control with Neurodesk on HPC

**Author**: Kelly G. Garner, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/garner-code"><img src="https://img.shields.io/badge/-Kelly_G._Garner-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

MRIQC (Magnetic Resonance Image Quality Control) is a tool for extracting quality metrics from MRI scans and generating comprehensive quality control reports. This tutorial guides you through using MRIQC with Neurodesk on a high-performance computing (HPC) cluster. You will learn how to run quality assessments at both the participant level (individual images) and group level (summary statistics), configure MRIQC for HPC environments, and interpret quality control reports.

:::{admonition} Learning Objectives
:class: tip

After completing this tutorial, you will be able to:

- Open the MRIQC container from the Neurodesk application menu
- Run MRIQC at the participant level to assess individual images
- Run MRIQC at the group level to generate summary quality reports
- Configure MRIQC for HPC use with appropriate resource parameters
- Interpret MRIQC quality metrics and identify problematic scans

:::

## Citation and Resources

**MRIQC**
: Esteban, O., et al. (2017). MRIQC: Advancing the automatic prediction of image quality in MRI from unseen sites. *PLOS ONE*, 12(9), e0184661. https://doi.org/10.1371/journal.pone.0184661

**Official Documentation**
: https://mriqc.readthedocs.io/

## Prerequisites

:::{admonition} Requirements
:class: warning

- Data must be in BIDS (Brain Imaging Data Structure) format
- Neurodesk running on your HPC system or local machine
- Sufficient disk space for quality metrics (~1-5 GB per participant)
- Access to HPC job scheduler (SLURM, PBS, etc.) if running on HPC
- Basic understanding of BIDS data structure

:::

### Setup Checklist

- [ ] My data is in BIDS format
- [ ] Neurodesk is installed and running
- [ ] I understand my HPC resource allocation limits
- [ ] I have sufficient disk space for outputs
- [ ] I know which MRI modalities I want to assess (T1w, T2w, fMRI, DWI, etc.)

## Section 1: Open MRIQC from Neurodesk

### Important: Home Directory Configuration on HPC

MRIQC has `$HOME` hardcoded to `/home/mriqc` in some versions. On certain HPC systems, this can cause permission or path issues. If you encounter home directory-related errors, run this command **before** launching Neurodesktop:

```bash
export neurodesk_singularity_opts="--home $HOME:/home"
```

This remaps the container's home directory to your actual home directory.

### Step 1: Launch Neurodesktop

1. If you are using Neurodesk in a Jupyter environment, click the **Neurodesktop** icon in the JupyterLab launcher
2. This will launch the Neurodesktop desktop environment

![Launching Neurodesktop from the JupyterLab launcher](/static/tutorials/functional_imaging/mriqc/launch_neurodesk.png)
*Launching Neurodesktop from the JupyterLab launcher.*

### Step 2: Access the Neurodesk Application Menu

1. Once Neurodesktop loads, look for the application menu
2. Click on the **Neurodesk** menu or icon

![The Neurodesk application menu](/static/tutorials/functional_imaging/mriqc/neurodesk_menu.png)
*The Neurodesk application menu.*

### Step 3: Navigate to MRIQC

1. In the Neurodesk menu, find **MRIQC**
2. Select the latest version available (usually at the bottom of the list)
3. Click to open MRIQC

![Selecting MRIQC from the Neurodesk menu](/static/tutorials/functional_imaging/mriqc/open_mriqc.png)
*Selecting MRIQC from the Neurodesk menu (latest version at the bottom).*

### Step 4: Verify Container is Ready

1. A terminal window will open showing the MRIQC container environment
2. You will see a bash prompt indicating the container is ready for commands

![MRIQC container terminal ready to use](/static/tutorials/functional_imaging/mriqc/mriqc_bash.png)
*The MRIQC container terminal open and ready to use.*

## Section 2: Run Participant-Level MRIQC

Participant-level analysis extracts quality metrics from individual participant images.

### Step 1: Set Thread Limits (HPC Only)

If running on an HPC system, set the number of threads to prevent exceeding resource limits:

```bash
export ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS=6
```

Adjust the value based on your available cores and HPC allocation.

### Step 2: Run MRIQC at Participant Level

Here is a typical MRIQC participant-level command:

```bash
mriqc /path/to/your/data \
      /path/to/your/data/derivatives \
      participant \
      --participant-label 01 \
      --work-dir /path/to/work/directory \
      --nprocs 6 --mem_gb 10000 \
      -v
```

### Step 3: Understanding Key Parameters

| Parameter | Purpose | Example |
|-----------|---------|----------|
| `/path/to/your/data` | Input BIDS directory | `/home/user/mydata` |
| `/path/to/your/data/derivatives` | Output derivatives directory | `/home/user/mydata/derivatives` |
| `participant` | Analysis level (participant or group) | `participant` |
| `--participant-label` | Specific participant(s) to process | `01` or `01 02 03` |
| `--work-dir` | Working directory for temporary files | `/scratch/mriqc_work` |
| `--nprocs` | Number of processing threads (HPC) | `6` |
| `--mem_gb` | Memory limit in GB (HPC) | `10` |
| `-v` | Verbose output (more logging) | (flag) |

### Step 4: For Local Processing

If running on a local machine (not HPC), you can omit the `--nprocs` and `--mem_gb` parameters:

```bash
mriqc /path/to/your/data \
      /path/to/your/data/derivatives \
      participant \
      --participant-label 01 \
      --work-dir /tmp/mriqc_work \
      -v
```

### Step 5: Process Multiple Participants

To process multiple participants in sequence:

```bash
mriqc /path/to/your/data \
      /path/to/your/data/derivatives \
      participant \
      --participant-label 01 02 03 04 05 \
      --work-dir /path/to/work/directory \
      --nprocs 6 --mem_gb 10000 \
      -v
```

### Step 6: Participant-Level Output

Once processing completes, you will find:

- **Individual quality metrics**: Stored in `/derivatives/sub-XX/anat/` and `/derivatives/sub-XX/func/` as JSON files
- **Individual reports**: HTML quality reports for each participant and scan
- **Derivative images**: Preprocessed images used for quality assessment

## Section 3: Run Group-Level MRIQC

Group-level analysis generates summary statistics and visualisations across all participants.

### Important: Complete Participant-Level Processing First

**The group-level analysis requires that participant-level analysis has already completed.** The `derivatives` folder must already contain the results from participant processing.

### Step 1: Run MRIQC at Group Level

Once all participants have been processed, run:

```bash
mriqc /path/to/your/data \
      /path/to/your/data/derivatives \
      group \
      -w /path/to/work/directory \
      --nprocs 6 --mem_gb 10000 \
      -v
```

### Step 2: Understanding Group-Level Parameters

Most parameters are the same as participant level:

| Parameter | Purpose |
|-----------|----------|
| `group` | Specifies group-level analysis (instead of `participant`) |
| `-w` | Working directory for temporary files |
| `--nprocs` | Number of processing threads |
| `--mem_gb` | Memory limit in GB |

### Step 3: For Local Processing

When running locally, omit resource limit parameters:

```bash
mriqc /path/to/your/data \
      /path/to/your/data/derivatives \
      group \
      -w /tmp/mriqc_work \
      -v
```

### Step 4: Group-Level Output

Group-level analysis generates:

- **Group metrics CSV**: Summary table of quality metrics across all participants
- **Group HTML report**: Interactive visualisations of quality metrics with statistical distributions
- **Metadata**: JSON files containing group-level analysis metadata

The main output file is typically named `group_T1w.html` and `group_T2w.html` (depending on your modalities).

### Step 5: Interpret Group-Level Results

The group-level HTML report includes:

1. **Distribution plots**: Show how each quality metric is distributed across your cohort
2. **Outlier detection**: Highlights participants with unusual quality metrics
3. **Correlation matrices**: Show relationships between different quality metrics
4. **Batch effects**: Identify systematic quality differences if data was acquired in different batches

Use this report to:
- Identify problematic scans for exclusion
- Understand quality variation in your sample
- Plan follow-up acquisition strategies

## Summary

In this tutorial, you have:

- Configured the MRIQC home directory for HPC compatibility
- Opened the MRIQC container from the Neurodesk application menu
- Ran MRIQC at the participant level to extract quality metrics from individual scans
- Configured MRIQC with appropriate HPC resource parameters
- Ran MRIQC at the group level to generate comprehensive quality control reports
- Learned how to process single and multiple participants
- Understood how to interpret group-level quality control results

You now have the knowledge to perform thorough quality control on your MRI data before preprocessing, helping you identify and handle problematic scans and ensure the overall quality of your dataset.

## See Also

- [fMRIPrep Tutorial](./fmriprep.ipynb) — Comprehensive preprocessing of functional MRI data
- [BIDS Conversion Essentials](../data_preparation/bids_conversion.ipynb) — Convert your data to BIDS format for use with MRIQC
- [MRIQC Official Documentation](https://mriqc.readthedocs.io/) — Comprehensive MRIQC documentation and metrics explanation
- [MRIQC on ReadTheDocs](https://mriqc.readthedocs.io/en/latest/) — Detailed information about quality metrics